<a href="https://colab.research.google.com/github/Nithya2405/ai-engineering-workbench/blob/main/AI_researcher_agent_app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# STEP 1: RESILIENT BINARY DOWNLOAD & ENV SETUP
# ==============================================================================
import os
import subprocess

print("🚀 Starting Professional Workbench Deployment...")

# Manual download with a different User-Agent to bypass 403 Forbidden
!curl -Lk https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip -o ngrok.zip
!unzip -o ngrok.zip
!chmod +x ./ngrok
!pip install -q pyngrok groq tavily-python streamlit

# ==============================================================================
# STEP 2: WRITE THE STREAMLIT APP FILE
# ==============================================================================
with open('researcher_agent_app.py', 'w') as f:
    f.write("""
import streamlit as st
from groq import Groq
from tavily import TavilyClient

st.set_page_config(page_title="AI Agent Workbench", layout="wide", page_icon="🔬")

st.markdown("<style>.main { background-color: #0e1117; } .stStatus { border-radius: 10px; }</style>", unsafe_allow_html=True)
st.title("🔬 Professional Agentic Research Workbench")
st.caption("Phase 2 Conclusion: Multi-Agent Orchestration & Evaluation")

with st.sidebar:
    st.header("⚙️ Configuration")
    g_api = st.text_input("Groq API Key", type="password")
    t_api = st.text_input("Tavily API Key", type="password")
    st.divider()
    st.info("Architecture: Llama-3.3-70B + Tavily + LLM-as-a-Judge")

def run_pipeline(query, g_key, t_key):
    client = Groq(api_key=g_key)
    tavily = TavilyClient(api_key=t_key)
    with st.status("👑 Supervisor: Orchestrating...", expanded=True) as s:
        st.write("Decomposing Task...")
        plan_prompt = f"Plan a research strategy for: {query}. Return a 1-sentence plan."
        plan = client.chat.completions.create(model='llama-3.3-70b-versatile', messages=[{'role': 'user', 'content': plan_prompt}]).choices[0].message.content
        st.write(f"**Plan:** {plan}")

        st.write("🕵️ Fetching Intelligence...")
        res = tavily.search(query=query, max_results=3)
        ctx = "\\n".join([r['content'] for r in res['results']])

        st.write("✍️ Synthesizing Report...")
        gen_prompt = f"Using this info: {ctx}, answer: {query}. Be concise."
        rpt = client.chat.completions.create(model='llama-3.3-70b-versatile', messages=[{'role': 'user', 'content': gen_prompt}]).choices[0].message.content
        s.update(label="✅ Success", state="complete")
    return rpt

def run_eval(query, rpt, g_key):
    client = Groq(api_key=g_key)
    with st.status("⚖️ Judge: Auditing Quality..."):
        eval_prompt = f"User asked: {query}. Agent replied: {rpt}. Grade 1-10 on accuracy. Format: SCORE: [X/10] | REASON: [text]"
        audit = client.chat.completions.create(model='llama-3.3-70b-versatile', messages=[{'role': 'system', 'content': eval_prompt}]).choices[0].message.content
    return audit

topic = st.text_input("Research Topic:", placeholder="e.g., Future of LLMOps")
if st.button("Execute Pipeline"):
    if not g_api or not t_api: st.error("Please enter keys in sidebar.")
    else:
        report = run_pipeline(topic, g_api, t_api)
        st.subheader("📑 Final Research Report")
        st.markdown(report)
        st.divider()
        st.subheader("⚖️ Quality Audit")
        st.info(run_eval(topic, report, g_api))
    """)

# ==============================================================================
# STEP 3: AUTHENTICATION & STABLE TUNNEL
# ==============================================================================
from pyngrok import ngrok, conf

# Insert your Token below
NGROK_TOKEN = "3Binv7Ace9CPEjCriA0dAX1Yipj_2ZmXPm4G3AwrEjbQTz149"

# Direct binary pathing to prevent install errors
py_conf = conf.PyngrokConfig(ngrok_path="./ngrok")
conf.set_default(py_conf)
ngrok.set_auth_token(NGROK_TOKEN)

# Clean up old processes
!pkill streamlit
!pkill ngrok

# Launch Streamlit
subprocess.Popen(["streamlit", "run", "researcher_agent_app.py", "--server.port", "8501", "--server.headless", "true"])

# Launch Stable Tunnel
public_url = ngrok.connect(8501, pyngrok_config=py_conf)
print(f"\n\n✅ DEPLOYMENT SUCCESSFUL!")
print(f"🔗 Access your Workbench here: {public_url}")
print("👉 Note: Click 'Visit Site' on the ngrok landing page.")